# 3. Feature Selection

After preprocessing, our dataset contains 137 features. However, not all of them may be useful for predicting repeat purchases.

We will apply feature-selection methods to identify relevant features and evaluate whether a smaller feature set can maintain or improve model performance.

# Import Libraries

In [38]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import roc_auc_score

# 3.1 Load Preprocessed Data

We load the datasets prepared in the preprocessing notebook so we can apply feature selection to the cleaned data.

In [39]:
%run "02_Clean_Preprocessing.ipynb"

Number of incomplete observations: 65

Available information:
repeat_purchase_90d    65
customer_id            65
ID                     65
dtype: int64
Missing purchase dates: 65
Missing dates in incomplete observations: 65


,ID,prior_total_spend,spend_180d,spend_365d
263,10000300,3456.32,577.67,-3456.320000
1733,10002022,5129.20,264.00,-1230.720000
1744,10002033,2586.88,0.00,-167.180000
1829,10002134,753.51,0.00,-753.510000
2099,10002452,28625.97,2234.61,-2648.440000
2851,10003340,31123.69,0.00,-2169.910000
3686,10004321,3215.36,420.49,-1443.640000
3690,10004327,9615.39,591.02,-5133.040000
3841,10004511,58473.69,9560.84,-14740.290000
3893,10004574,10042.78,179.58,-4196.900000


Remaining inconsistent averages: 0
Number of rows: 6534

Missing values:
commercial_region      35
customer_since_year     0
dtype: int64
Training observations: 5024
Validation observations: 1298
Training shape: (5024, 137)
Validation shape: (1298, 137)
[]
Test shape: (986, 137)
Same number of features: True
All rows preserved: True
IDs preserved: True


# 3.2 Filter Method — SelectKBest

We applied SelectKBest with the ANOVA F-score to rank features according to their individual relationship with the target variable.

We initially selected the 30 highest-scoring features. The selector was fitted only on the training data to avoid data leakage.

In [40]:
# Get the feature names after preprocessing
feature_names = preprocessor.get_feature_names_out()

# Select the 30 highest-scoring features
selector = SelectKBest(score_func=f_classif, k=30)

X_train_selected = selector.fit_transform(
    X_train_processed, y_train
)

X_val_selected = selector.transform(X_val_processed)
X_test_selected = selector.transform(X_test_processed)

# Show the selected features and their scores
results = pd.DataFrame({
    "Feature": feature_names,
    "F-score": selector.scores_,
    "Selected": selector.get_support()
})

display(
    results[results["Selected"]]
    .sort_values("F-score", ascending=False)
    .head(15)
)

print("Original features:", X_train_processed.shape[1])
print("Selected features:", X_train_selected.shape[1])

,Feature,F-score,Selected
26,num__purchase_count_365d,663.520046,True
24,num__purchase_count_180d,648.296274,True
22,num__purchase_count_90d,629.820695,True
25,num__spend_180d,574.235418,True
23,num__spend_90d,459.278116,True
12,num__prior_purchase_count,450.786661,True
20,num__purchase_count_30d,409.431627,True
13,num__prior_total_spend,390.537912,True
40,cat__payment_type_Maybe Tomorrow Payment 9,310.020792,True
29,num__std_gap_missing,243.574825,True


Original features: 137
Selected features: 30


The highest-scoring features are mainly related to previous purchase frequency and recent spending, suggesting that purchasing history is important for predicting repeat purchases.

We initially chose `k=30` to reduce the number of features while keeping a reasonable amount of information. We used the same number of features for RFE to allow a direct comparison. This value was chosen for experimentation and was not optimized.

However, we cannot conclude that selecting 30 features improves model performance without comparing it with a model using all features.

# 3.3 Comparing Feature Selection Performance

To evaluate whether feature selection improves performance, we compare two Logistic Regression models: one using all features and another using the 30 features selected by SelectKBest.

We use the F1-score on the validation set to compare their performance. Both models are trained and evaluated using the same data split.

#### • Compare All Features vs Selected Features

In [41]:
# Model using all features
model_all = LogisticRegression(max_iter=1000, random_state=42)
model_all.fit(X_train_processed, y_train)

pred_all = model_all.predict(X_val_processed)
f1_all = f1_score(y_val, pred_all)


# Model using the 30 selected features
model_selected = LogisticRegression(max_iter=1000, random_state=42)
model_selected.fit(X_train_selected, y_train)

pred_selected = model_selected.predict(X_val_selected)
f1_selected = f1_score(y_val, pred_selected)


# Compare the results
comparison = pd.DataFrame({
    "Feature Set": ["All Features", "SelectKBest"],
    "Number of Features": [
        X_train_processed.shape[1],
        X_train_selected.shape[1]
    ],
    "Validation F1": [f1_all, f1_selected]
})


display(comparison.round(4))

,Feature Set,Number of Features,Validation F1
0,All Features,137,0.7542
1,SelectKBest,30,0.7537


The model using all 137 features achieved a slightly higher F1-score (0.7542) than the model using the 30 features selected by SelectKBest (0.7537).

However, the difference is very small. This suggests that a smaller feature set may achieve similar predictive performance while reducing model complexity.

We will compare additional feature-selection methods before choosing our final feature set.

#### • Baseline Comparison

We calculated a simple baseline by always predicting that the customer will make another purchase.

The baseline achieved an F1-score of 0.7525 and a ROC-AUC of 0.5. This provides a reference for evaluating whether our models perform better than a simple prediction rule.

In [42]:
baseline_predictions = np.ones(len(y_val), dtype=int)

baseline_f1 = f1_score(y_val, baseline_predictions)

print("Baseline F1:", baseline_f1)

baseline_auc = roc_auc_score(y_val, baseline_predictions)

print("Baseline ROC-AUC:", baseline_auc)

Baseline F1: 0.7525228255646323
Baseline ROC-AUC: 0.5


# 3.4 Wrapper Feature Selection — RFE

We applied Recursive Feature Elimination (RFE) with Logistic Regression, following the Wrapper method introduced in Week 4.

RFE repeatedly fits a model and removes the least important features based on their coefficients.

We initially selected 30 features to compare this method with SelectKBest. Feature selection was fitted only on the training data to avoid data leakage.

#### • Apply Recursive Feature Elimination

In [43]:
rfe_selector = RFE(
    estimator=LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    n_features_to_select=30,
    step=5
)

# Fit RFE on training data only
X_train_rfe = rfe_selector.fit_transform(
    X_train_processed, y_train
)

# Apply the same selection to validation and test
X_val_rfe = rfe_selector.transform(X_val_processed)
X_test_rfe = rfe_selector.transform(X_test_processed)

# Display selected features
rfe_features = feature_names[rfe_selector.get_support()]

print("Original features:", X_train_processed.shape[1])
print("Selected features:", X_train_rfe.shape[1])

display(pd.DataFrame({"Selected Features": rfe_features}))

Original features: 137
Selected features: 30


,Selected Features
0,num__purchase_count_90d
1,num__purchase_count_365d
2,cat__payment_type_Maybe Tomorrow Payment 8
3,cat__payment_type_Maybe Tomorrow Payment 9
4,cat__payment_type_infrequent_sklearn
5,cat__salesperson_VLOOKUP Master 6
6,cat__salesperson_infrequent_sklearn
7,cat__transport_method_Maybe Tomorrow Delivery 11
8,cat__transport_method_Maybe Tomorrow Delivery 17
9,cat__transport_method_Maybe Tomorrow Delivery 18


#### • Evaluate RFE Feature Selection

In [44]:
model_rfe = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_rfe.fit(X_train_rfe, y_train)

pred_rfe = model_rfe.predict(X_val_rfe)

f1_rfe = f1_score(y_val, pred_rfe)

# Compare the three approaches
comparison = pd.DataFrame({
    "Feature Set": [
        "All Features",
        "SelectKBest",
        "RFE"
    ],
    "Number of Features": [
        X_train_processed.shape[1],
        X_train_selected.shape[1],
        X_train_rfe.shape[1]
    ],
    "Validation F1": [
        f1_all,
        f1_selected,
        f1_rfe
    ]
})

display(comparison)

,Feature Set,Number of Features,Validation F1
0,All Features,137,0.754232
1,SelectKBest,30,0.753662
2,RFE,30,0.767664


RFE achieved a validation F1-score of 0.7677, higher than SelectKBest (0.7537) and the model using all features (0.7542).

This suggests that RFE performed better according to F1-score while reducing the number of features from 137 to 30. However, we will also compare ROC-AUC and examine the selected features before making our final decision.

# 3.5 Embedded Feature Selection — L1 Logistic Regression

We applied SelectFromModel with L1 Logistic Regression, following the Embedded method covered in Week 4.

L1 regularization can reduce less useful feature coefficients to zero, allowing us to select the remaining features.

The selector is fitted only on training data to avoid data leakage.

#### • Apply Embedded Feature Selection

In [45]:
embedded_selector = SelectFromModel(
    LogisticRegression(
        l1_ratio=1,
        solver="liblinear",
        C=0.1,
        max_iter=1000,
        random_state=42
    ),
    threshold=1e-8
)

# Fit the selector on training data
X_train_embedded = embedded_selector.fit_transform(
    X_train_processed, y_train
)

# Apply the same selection to validation and test data
X_val_embedded = embedded_selector.transform(X_val_processed)
X_test_embedded = embedded_selector.transform(X_test_processed)

# Display selected features
embedded_features = feature_names[embedded_selector.get_support()]

print("Original features:", X_train_processed.shape[1])
print("Selected features:", X_train_embedded.shape[1])

display(pd.DataFrame({
    "Selected Features": embedded_features
}).head(15))

Original features: 137
Selected features: 48


,Selected Features
0,num__Total Liquido
1,num__Total Imp_
2,num__Total Portes
3,num__Total Desc_ Linha
4,num__Custo
5,num__N_ Detalhes
6,num__invoice_count_current_purchase
7,num__days_since_previous_purchase
8,num__customer_tenure_days
9,num__prior_purchase_count


#### • Evaluate Embedded Feature Selection

In [46]:
model_embedded = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_embedded.fit(X_train_embedded, y_train)

pred_embedded = model_embedded.predict(X_val_embedded)
f1_embedded = f1_score(y_val, pred_embedded)

# Compare all four approaches
comparison = pd.DataFrame({
    "Feature Set": [
        "All Features",
        "SelectKBest",
        "RFE",
        "Embedded (L1)"
    ],
    "Number of Features": [
        X_train_processed.shape[1],
        X_train_selected.shape[1],
        X_train_rfe.shape[1],
        X_train_embedded.shape[1]
    ],
    "Validation F1": [
        f1_all,
        f1_selected,
        f1_rfe,
        f1_embedded
    ]
})

display(comparison)

,Feature Set,Number of Features,Validation F1
0,All Features,137,0.754232
1,SelectKBest,30,0.753662
2,RFE,30,0.767664
3,Embedded (L1),48,0.756349


We used `C=0.1` to apply relatively strong L1 regularization, encouraging the model to set less useful feature coefficients to zero. This was an initial choice rather than an optimized value.

The Embedded L1 method selected 48 features and achieved a validation F1-score of 0.7563, slightly higher than the model using all features (0.7542).

Although RFE achieved a higher F1-score, we cannot conclude which method is preferable based on this metric alone. We will also compare ROC-AUC before choosing our final feature set.

# 3.6 Combining Strategies — Voting

We combined the Filter, Wrapper and Embedded methods using a voting strategy, as introduced in Week 4.

A feature is retained if at least two of the three methods select it. We then evaluate the selected features using Logistic Regression.

We compare the methods using both F1-score and ROC-AUC. F1 measures the balance between precision and recall, while ROC-AUC evaluates how well the model distinguishes the two classes across different classification thresholds.

#### • Apply Voting Feature Selection

In [47]:
# Count how many methods selected each feature
votes = (
    selector.get_support().astype(int) +
    rfe_selector.get_support().astype(int) +
    embedded_selector.get_support().astype(int)
)

# Keep features selected by at least 2 methods
vote_mask = votes >= 2

# Apply the selection to all datasets
X_train_vote = X_train_processed[:, vote_mask]
X_val_vote = X_val_processed[:, vote_mask]
X_test_vote = X_test_processed[:, vote_mask]

# Display the selected features
voting_features = feature_names[vote_mask]

print("Original features:", X_train_processed.shape[1])
print("Features selected by voting:", len(voting_features))

display(pd.DataFrame({
    "Selected Features": voting_features,
    "Number of Votes": votes[vote_mask]
}))

Original features: 137
Features selected by voting: 28


,Selected Features,Number of Votes
0,num__days_since_previous_purchase,2
1,num__prior_purchase_count,2
2,num__prior_total_spend,2
3,num__prior_mean_purchase_value,2
4,num__prior_std_purchase_value,2
5,num__prior_max_purchase_value,2
6,num__prior_mean_gap_days,2
7,num__purchase_count_30d,2
8,num__spend_30d,2
9,num__purchase_count_90d,3


#### • Evaluate Voting Feature Selection

In [48]:
model_vote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_vote.fit(X_train_vote, y_train)

pred_vote = model_vote.predict(X_val_vote)

f1_vote = f1_score(y_val, pred_vote)

# Compare all feature-selection strategies
comparison = pd.DataFrame({
    "Feature Set": [
        "All Features",
        "SelectKBest",
        "RFE",
        "Embedded (L1)",
        "Voting"
    ],
    "Number of Features": [
        X_train_processed.shape[1],
        X_train_selected.shape[1],
        X_train_rfe.shape[1],
        X_train_embedded.shape[1],
        X_train_vote.shape[1]
    ],
    "Validation F1": [
        f1_all,
        f1_selected,
        f1_rfe,
        f1_embedded,
        f1_vote
    ]
})
# • Compare F1 and ROC-AUC

comparison["Validation ROC-AUC"] = [
    roc_auc_score(y_val, model_all.predict_proba(X_val_processed)[:, 1]),
    roc_auc_score(y_val, model_selected.predict_proba(X_val_selected)[:, 1]),
    roc_auc_score(y_val, model_rfe.predict_proba(X_val_rfe)[:, 1]),
    roc_auc_score(y_val, model_embedded.predict_proba(X_val_embedded)[:, 1]),
    roc_auc_score(y_val, model_vote.predict_proba(X_val_vote)[:, 1])
]

display(comparison.round(4))

,Feature Set,Number of Features,Validation F1,Validation ROC-AUC
0,All Features,137,0.7542,0.7983
1,SelectKBest,30,0.7537,0.7889
2,RFE,30,0.7677,0.7924
3,Embedded (L1),48,0.7563,0.8059
4,Voting,28,0.7642,0.7950


The Voting strategy selected 28 features and achieved an F1-score of 0.7642 and a ROC-AUC of 0.7950.

RFE achieved the highest F1-score (0.7677), while Embedded L1 achieved the highest ROC-AUC (0.8059).

The differences between methods are relatively small, so no single strategy clearly outperforms the others across both metrics.

# 3.7 Evaluating Identifier-Like Features

Although Embedded L1 selected some salesperson and transport method categories, these variables may capture patterns that are specific to our dataset.

To investigate their usefulness, we compare the original L1 feature set with a reduced version that excludes these categories.

We evaluate both using Logistic Regression, F1-score and ROC-AUC.

#### • Remove Identifier-Like Categorical Features

In [49]:
# Get features selected by Embedded L1
l1_features = feature_names[embedded_selector.get_support()]

# Identify salesperson and transport method features
remove_mask = np.array([
    name.startswith(("cat__salesperson_", "cat__transport_method_"))
    for name in l1_features
])

# Keep the remaining features
keep_mask = ~remove_mask

# Create reduced datasets
X_train_l1_reduced = X_train_embedded[:, keep_mask]
X_val_l1_reduced = X_val_embedded[:, keep_mask]
X_test_l1_reduced = X_test_embedded[:, keep_mask]

# Verify results
print("Features to remove:", remove_mask.sum())
print("Original L1 features:", len(l1_features))
print("Reduced L1 features:", keep_mask.sum())

Features to remove: 5
Original L1 features: 48
Reduced L1 features: 43


#### • Evaluate Reduced L1 Feature Set

In [50]:
model_l1_reduced = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model_l1_reduced.fit(X_train_l1_reduced, y_train)

# Predict validation data
pred_l1_reduced = model_l1_reduced.predict(X_val_l1_reduced)

# Calculate performance metrics
f1_l1_reduced = f1_score(y_val, pred_l1_reduced)

auc_l1_reduced = roc_auc_score(
    y_val,
    model_l1_reduced.predict_proba(X_val_l1_reduced)[:, 1]
)

# Compare with original Embedded L1
comparison_l1 = pd.DataFrame({
    "Feature Set": [
        "Original Embedded L1",
        "L1 without Salesperson and Transport"
    ],
    "Number of Features": [
        X_train_embedded.shape[1],
        X_train_l1_reduced.shape[1]
    ],
    "Validation F1": [
        f1_embedded,
        f1_l1_reduced
    ],
    "Validation ROC-AUC": [
        roc_auc_score(
            y_val,
            model_embedded.predict_proba(X_val_embedded)[:, 1]
        ),
        auc_l1_reduced
    ]
})

display(comparison_l1.round(4))

,Feature Set,Number of Features,Validation F1,Validation ROC-AUC
0,Original Embedded L1,48,0.7563,0.8059
1,L1 without Salesperson and Transport,43,0.7552,0.8063


#### • Conclusion

Our identifier-like feature experiment only removed the salesperson and transport method categories selected by L1. Other categorical variables, such as document series and commercial zone, were retained but may also capture dataset-specific patterns. Their generalization should be investigated in future work.

Removing salesperson and transport method features reduced the Embedded L1 feature set from 48 to 43 features.

The validation F1-score decreased slightly from 0.7563 to 0.7552, while ROC-AUC increased from 0.8059 to 0.8063.

Since performance remained almost unchanged, we prefer the reduced feature set because it is simpler and avoids potentially dataset-specific categories.

# 3.8 Final Feature Selection Decision

We compared Filter (SelectKBest), Wrapper (RFE), Embedded (L1) and Voting feature-selection methods, following the approaches covered in Week 4.

RFE achieved the highest validation F1-score (0.7677), while the original Embedded L1 method achieved the highest ROC-AUC (0.8059) among the initial feature-selection strategies.

We selected **Embedded L1 without salesperson and transport method features** as our preferred strategy, retaining 43 of the original 137 features.

This feature set achieved a validation F1-score of 0.7552 and a ROC-AUC of 0.8063. Although its F1-score was lower than RFE's, it achieved the highest ROC-AUC among the feature sets evaluated and retained relevant purchasing-history variables.

We excluded `ID` and `customer_id` because they are identifiers, and `random_noise` because it was not expected to contain useful predictive information. Removing the selected salesperson and transport method categories also reduces reliance on potentially dataset-specific patterns.

Since our results are based on a single validation split, the final feature set should be reassessed during the modelling stage.

Although our final feature set achieved the highest ROC-AUC, its F1-score (0.7552) was only slightly higher than the baseline (0.7525). This suggests that further improvements may be needed during model training and optimization.

#### • Display Final Selected Features

In [51]:
final_features = l1_features[keep_mask]

print("Number of selected features:", len(final_features))

display(pd.DataFrame({
    "Selected Features": final_features
}))

Number of selected features: 43


,Selected Features
0,num__Total Liquido
1,num__Total Imp_
2,num__Total Portes
3,num__Total Desc_ Linha
4,num__Custo
5,num__N_ Detalhes
6,num__invoice_count_current_purchase
7,num__days_since_previous_purchase
8,num__customer_tenure_days
9,num__prior_purchase_count
